# Quick Boys Stadium Expansion

This notebook investigades a Multi-Objective Design Optimisation (MODO) model for an expansion of the quick boys stadium in Katwijk. The following information can be found:

- Define design variables and bounds
- Specify multiple constraints and objective functions
- Run a Algorithm to calculate optimization
- Visualize preference functions and optimization results





In [13]:
# Import libraries
import matplotlib.pyplot as plt
import numpy as np
from scipy.interpolate import pchip_interpolate
from scipy.optimize import minimize  

# Import local module for genetic algorithm
#from genetic_algorithm_pfm import GeneticAlgorithm 

## Variables and Bounds


| Variable | Description | Unit | Type |
|----------|-------------|------|------|
| `x1` | Number of seats | - | Discrete |
| `x2` | Number of parking places | - | Discrete |
| `x3` | Width of tribune | m | Continuous |
| `x4` | Length of tribune | m | Continuous | 
| `x5` | Number of rows | - | Discrete | 
| `x6` | Investment | € | Continuous | 

In [14]:
# set bounds for all variables
b1 = [350, 900]      # x1
b2 = [250, 600]      # x2
b3 = [0, 25]         # x3
b4 = [0, 90]         # x4
b5 = [1, 100]        # x5  
b6 = [0, 10000000]   # x6
bounds = [b1, b2, b3, b4, b5, b6]

## Constraints

bla bla bla

In [15]:
#Constraint functions

## Objectives

| Stakeholder | Objective | Unit |
|-------------|-----------|------|
| Residents | Noise | dB |
| Environment | Environmental impact | CO2 |
| Quick Boys | Rendement on Investment | €/year |
| Fans | Comfort | m<sup>3</sup>/pp |
| Players | Player Facilities | m<sup>2</sup> |
| Authorities | Fire/Emergency Access | s|

In [ ]:
"""
x1: Number of seats
x2: Number of parking spaces
x3: Width tribune [m]
x4: Length tribune [m]
x5: Number of rows
x6: Investment [€]
"""

def cost(x1, x2, x3, x4, x5, x6):
    """
    This function defines the cost of the rebuilding of the stadium.
    Output: Cost in euros

    """
    cost_seat = 500       # Cost per seat
    cost_parking = 1000   # Cost per parking space
    cost_tribune = 500    # Cost per m^2 of the tribune
    return cost_seat*x1 + cost_parking*x2 + cost_tribune*x3*x4

def obj_func_1(x1, x2, x3, x4, x5, x6):
    """ 
    This objective function defines the noise output from the stadium.
    Output: Noise in dB arriving at the residents.

    This function is currently outputting an index
    """
    noise_seat = 1.0    # Noise per seat
    noise_parking = 0.5 # Noise per parking place
    return noise_seat*x1 + noise_parking*x2

def obj_func_2(x1, x2, x3, x4, x5, x6):
    """ 
    This objective function defines the environmental impact in CO2.
    Output: kg CO2
    """
    co2_parking = 1.0   # kg CO2 per parking space
    co2_tribune = 1.0   # kg CO2 per m^2 of the tribune
    return  co2_parking*x2 + co2_tribune*x3*x4

def obj_func_3(x1, x2, x3, x4, x5, x6):
    """ 
    This objective function defines the return on investment.
    Output: €
    """
    rev_ticket = 30     # Ticket price
    rev_sponsor = 5     # Sponsoring
    rev_bar = 10        # Consumptions
    occupancy = 0.75
    games = 17          # Number of games per year
    revenue = rev_ticket*rev_sponsor*rev_bar*occupancy*games*x1
    return revenue / cost(x1, x2, x3, x4, x5, x6)

def obj_func_4(x1, x2, x3, x4, x5, x6):
    """ 
    This objective function defines the comfort of players.
    Output: rating
    """
    return (x3*x4)/x1 + x3/x5

def obj_func_5(x1, x2, x3, x4, x5, x6):
    """ 
    This objective function defines the player facilities.
    Output: rating
    """
    A_player = 0.1      # Area of the tribune that is dedicated to players
    return A_player*x3*x4

def obj_func_6(x1, x2, x3, x4, x5, x6):
    """ 
    This objective function defines m^2 per spectator for emergency access.
    Output: m^2 / supporter
    """
    A_seat = 0.5        # Area of a seat
    return (x3*x4 - A_seat*x1) / x1


In [17]:
objectives = [
    (obj_func_1, "Noise",                   "dB",         "Residents"),
    (obj_func_2, "Environmental impact",    "CO2",        "Environment"),
    (obj_func_3, "Rendement on Investment", "€/year",     "Quick Boys"),
    (obj_func_4, "Comfort",                 "m^3/year",   "Fans"),
    (obj_func_5, "Player Facilities",       "m^2",        "Players"),
    (obj_func_6, "Fire/Emergency Access",   "s",          "Authorities")
]

Evaluate min and max points per function

In [18]:
# Finding min and max for each objective using scipy's minimize function, starting from the midpoint of the bounds
objective_minmax = {} # Dictionary to store the min and max values for each objective       
midpoints = [np.mean(b) for b in bounds]

for i, (obj_func, name, unit, stakeholder) in enumerate(objectives):
    wrapped = lambda x, sign=1: sign * obj_func(*x)  # obj_func accepts a single array-like X
    
    min_val =  minimize(wrapped, x0=midpoints, bounds=bounds, method='L-BFGS-B').fun
    max_val = -minimize(lambda x: wrapped(x, sign=-1), x0=midpoints, bounds=bounds, method='L-BFGS-B').fun

    objective_minmax[name] = min_val, max_val
    print(f"  Objective {i+1}  :    min = {min_val:>15,.1f}  {unit:<12}    max = {max_val:>15,.1f}  {unit}")


  Objective 1  :    min =           475.0  dB              max =         1,200.0  dB
  Objective 2  :    min =           250.0  CO2             max =         2,850.0  CO2
  Objective 3  :    min =             0.8  €/year          max =            11.9  €/year
  Objective 4  :    min =             0.0  m^3/year        max =            31.4  m^3/year
  Objective 5  :    min =             0.0  m^2             max =           225.0  m^2
  Objective 6  :    min =            -0.5  s               max =             5.9  s
